# P3 — Los caminos son mensajes

El puente entre P1 (over-squashing) y P2 (conteo de caminos). Tras `g` capas, un mensaje de `i` llega a `j`
**una vez por cada camino de longitud `g`**. Así que el conteo `n_g(i,j)` es exactamente *cuántas copias de la
información de `i` llegan a `j`* — todas apretadas en el único vector de `j`.

**5 figuras:** la pila de mensajes vs capacidad · explosión · barras crudo-vs-deduplicado · los dos operadores
· caminos redundantes fusionándose.

In [ ]:
import torch, numpy as np
from oversquash import viz
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash.ideal_bridge import walk_operators
K, M = 5, 4
def msgs_at(depth, op='raw'):
    dd = make_bottleneck_graph(K, M, depth, torch.Generator().manual_seed(0))
    raw, eff = walk_operators(dd.edge_index.numpy(), dd.num_nodes, max_length=depth+1)
    t = int(dd.root_mask.nonzero()[0])
    return (raw if op=='raw' else eff)[depth][:, t].sum()

## Figura 1 — mensajes apilándose más allá de la capacidad del vector

Cada barra es el número de mensajes que llegan al objetivo. La línea punteada es la capacidad del vector. Más
allá, la barra se vuelve roja — esos mensajes no caben. *Ese* es el apretujamiento.

In [ ]:
depths = [1,2,3,4]
raw_msgs = [int(msgs_at(d, 'raw')) for d in depths]
viz.explain_message_stack(raw_msgs, [f'prof {d}' for d in depths], capacity=10,
                          title='mensajes en el objetivo vs. capacidad del vector', xlabel='',
                          ylabel='mensajes en el vector del objetivo')

## Figura 2 — la explosión, en escala logarítmica

Los mismos conteos, eje log: llegan `K·M^profundidad` copias. El tamaño del vector nunca crece para igualarlo.

In [ ]:
viz.plot_message_explosion(depths, raw_msgs, 'mensajes que llegan al objetivo', 'profundidad', 'mensajes (log)')

## Figura 3 — pero muchos mensajes son redundantes

En nuestro cuello de botella, los nodos del medio son intercambiables, así que muchos caminos llevan el
**mismo** contenido. El cociente `kQ/I` fusiona caminos equivalentes: el conteo colapsa de `K·M^profundidad` a
~`K`.

In [ ]:
eff_msgs = [int(msgs_at(d, 'eff')) for d in depths]
viz.plot_raw_vs_eff(depths, raw_msgs, eff_msgs, 'mensajes crudos vs de-duplicados', 'profundidad',
                    'mensajes (log)', legend=('crudo  (A^g)', 'efectivo  (kQ/I)'))

## Figura 4 — las dos rutas redundantes, lado a lado

Dos rutas intercambiables que llevan el mismo contenido. `kQ/I` reconoce que son equivalentes y las cuenta una
vez. (Dibujado en un diamante pequeño para mayor claridad.)

In [ ]:
dia = [(0,1),(0,2),(1,3),(2,3)]; pos = {0:(0,0),1:(1,1),2:(1,-1),3:(2,0)}
viz.explain_paths_highlighted(dia, pos, [[(0,1),(1,3)],[(0,2),(2,3)]], 4, 0, 3,
                              title='dos rutas redundantes -> kQ/I conserva una', path_fmt='ruta {i}')

## Figura 5 — los operadores: crudo vs de-duplicado

`A^g` (conteo crudo) vs `M_g` (kQ/I de-duplicado). Mismo soporte, muchas menos copias a la derecha.

**El giro (demostrado en P4–P5):** fusionar es el arreglo *equivocado* — descarta señal. El arreglo correcto
es conservar todos los caminos pero **aprender cuánto pesar cada uno** (atención).

In [ ]:
dd = make_bottleneck_graph(K, M, 3, torch.Generator().manual_seed(0))
raw, eff = walk_operators(dd.edge_index.numpy(), dd.num_nodes, max_length=4)
viz.plot_two_heatmaps(raw[3], eff[3], titles=('A^4 (crudo)', 'M_4 (kQ/I, de-duplicado)'))